In [ ]:
## Bloco de codigo para instalar versao especifica dos pacotes
##pip install pandas==1.5.3
##pip install numpy==1.24.3

##### Carregando pacotes

In [1]:
# Pacotes de manipulacao

import pandas as pd
import numpy as np
import os

# Pacotes de visualizacao
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 1.5.3
numpy: 1.24.3


## Carregando databases

#### Base Dados Cadastrais

In [2]:
## Carregando todos arquivos em parquet de uma pasta

path = 'database/raw/base_score_bureau_movel/base_score_bureau_movel/'

all_files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.parquet')]
df_list = [pd.read_parquet(f, engine='pyarrow') for f in all_files]

df_bureau = pd.concat(df_list, ignore_index=True)

In [3]:
df_bureau.head()

,SAFRA,FLAG_INSTALACAO,FPD,PROD,flag_mig2,SCORE_01,SCORE_02,NUM_CPF
0,202410,1,0,CMV,PRE,562,636,ZZZZZX7XWY8
1,202410,1,1,CMV,PRE,546,518,ZZZZZX88YXY
2,202410,1,0,CMV,PRE,621,750,ZZZZZYT7XYT
3,202410,1,1,CMV,PRE,609,679,ZZZZZNTXY9Z
4,202410,1,0,CMV,PRE,621,722,ZZZZZ79ZXUX


In [14]:
df_bureau.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1290526 entries, 0 to 1290525
Data columns (total 8 columns):
 #   Column           Non-Null Count    Dtype 
---  ------           --------------    ----- 
 0   SAFRA            1290526 non-null  object
 1   FLAG_INSTALACAO  1290526 non-null  int32 
 2   FPD              1290526 non-null  object
 3   PROD             1290526 non-null  object
 4   flag_mig2        1290526 non-null  object
 5   SCORE_01         1281087 non-null  object
 6   SCORE_02         1289950 non-null  object
 7   NUM_CPF          1290526 non-null  object
dtypes: int32(1), object(7)
memory usage: 73.8+ MB


### Funções

Bloco destinado a criação das funcoes que serao utilizados ao longo deste notebook

In [4]:
# Metadados referente ao conjunto de dados
def generate_metadata(dataframe):
    """
    Gera um dataframe contendo metadados das colunas do dataframe fornecido.

    :param dataframe: DataFrame para o qual os metadados serão gerados.
    :return: DataFrame contendo metadados.
    """

    # Coleta de metadados básicos
    metadata = pd.DataFrame({
        'nome_variavel': dataframe.columns,
        'tipo': dataframe.dtypes,
        'qt_nulos': dataframe.isnull().sum(),
        'percent_nulos': round((dataframe.isnull().sum() / len(dataframe))* 100,2),
        'cardinalidade': dataframe.nunique(),
    })
    metadata=metadata.sort_values(by='percent_nulos',ascending=True)
    metadata = metadata.reset_index(drop=True)

    return metadata

In [19]:
def histplots_var_num(dataframe, bins=30):
    """
    Plota histogramas para todas as variáveis numéricas do dataframe fornecido
    em um painel com 3 gráficos por linha.

    :param dataframe: DataFrame para o qual os histogramas serão gerados.
    :param bins: Número de bins do histograma (default = 30).
    """
    # Seleciona apenas colunas numéricas
    numeric_columns = dataframe.select_dtypes(include=['float64', 'int64']).columns

    # Define o número de linhas com base no número de colunas numéricas
    nrows = len(numeric_columns) // 3 + (len(numeric_columns) % 3 > 0)

    # Inicializa o painel de gráficos
    fig, axes = plt.subplots(nrows=nrows, ncols=3, figsize=(14, nrows * 4))

    # Ajusta o layout
    plt.tight_layout(pad=4)

    # Configura estilo
    sns.set_style("whitegrid")

    # Plota histogramas para cada coluna numérica
    for i, column in enumerate(numeric_columns):
        sns.histplot(
            data=dataframe,
            x=column,
            bins=bins,
            kde=True,
            ax=axes[i // 3, i % 3],
            color="skyblue"
        )
        axes[i // 3, i % 3].set_title(
            f'{column}',
            fontdict={'fontsize': 14, 'fontweight': 'bold'}
        )
        axes[i // 3, i % 3].set_ylabel('')

    # Remove gráficos vazios (se houver)
    for j in range(i + 1, nrows * 3):
        fig.delaxes(axes.flatten()[j])

    # Adiciona título principal
    fig.suptitle(
        "Análise descritiva - Histogramas",
        fontsize=20,
        fontweight='bold',
        y=1.00
    )

### Analise dos dados

In [5]:
generate_metadata(df_bureau)

,nome_variavel,tipo,qt_nulos,percent_nulos,cardinalidade
0,SAFRA,object,0,0.00,6
1,FLAG_INSTALACAO,object,0,0.00,1
2,FPD,object,0,0.00,2
3,PROD,object,0,0.00,1
4,flag_mig2,object,0,0.00,1
5,NUM_CPF,object,0,0.00,1272095
6,SCORE_02,object,576,0.04,585
7,SCORE_01,object,9439,0.73,298


Olhando os metadados de cada coluna, ja podemos observar que:
- A base esta praticamente toda preenchida, possuindo apenas alguns nulos nas colunas *SCORE_01* e *SCORE_02*
- A coluna *FLAG_INSTALACAO* possui apenas um valor em todo o dataset
- A coluna *PROD* também possui apenas um valor em todo o dataset
- Todas as colunas estao com o data type object

### Analise univariada

Antes de seguirmos com a estruturação da tabela iremos analisar como os dados estão distribuidos.
Seguiremos com a ordem das colunas com o menor para o maior percentual de nulos.

#### 01 - Safra

In [6]:
df_bureau['SAFRA'].value_counts().sort_index()

202410    203828
202411    227176
202412    227985
202501    221002
202502    203139
202503    207396
Name: SAFRA, dtype: int64

In [7]:
print('Inicio da safra: ', df_bureau['SAFRA'].min())
print('Fim da safra: ', df_bureau['SAFRA'].max())

Inicio da safra:  202410
Fim da safra:  202503


Para o processo de ETL:
- Nome: Safra
- Transformar coluna em INT
- Criar duas novas colunas: Ano e Mes

#### 02 - FLAG_INSTALACAO

In [8]:
df_bureau['FLAG_INSTALACAO'].value_counts()

1    1290526
Name: FLAG_INSTALACAO, dtype: int64

In [11]:
# Iremos transformar essa coluna em tipo numerico
df_bureau['FLAG_INSTALACAO'] = df_bureau['FLAG_INSTALACAO'].astype('int')

Para o processo de ETL:
- Nome: IsSetup ou IsInstallation
- Tipo de dado: boolean

#### 03 - PROD

In [13]:
df_bureau['PROD'].value_counts()

CMV    1290526
Name: PROD, dtype: int64

Para o processo de ETL:
- Nome: ProductDescription
- Tipo de dado: Varchar

Possuimos apenas clientes com o tipo de produto CMV

#### 04 - flag_mig2

In [16]:
df_bureau['flag_mig2'].value_counts()

PRE    1290526
Name: flag_mig2, dtype: int64

Para o processo de ETL:
- Nome: ProductMigration
- Tipo de dado: Varchar

Todos os clientes desta base estao vindo de planos pre pagos

#### 05 - SCORE_01

In [18]:
# Transformando a variavel SCORE_01 em numerica
df_bureau['SCORE_01'] = pd.to_numeric(df_bureau['SCORE_01'], errors='coerce')

In [21]:
df_bureau['SCORE_01'].describe()

count    1.281087e+06
mean     5.869004e+02
std      5.748067e+01
min      0.000000e+00
25%      5.540000e+02
50%      5.870000e+02
75%      6.210000e+02
max      7.780000e+02
Name: SCORE_01, dtype: float64

Para o processo de ETL:
- Nome: Score01
- Tipo de dado: float

#### 06 - SCORE_02

In [22]:
# Transformando a variavel SCORE_01 em numerica
df_bureau['SCORE_02'] = pd.to_numeric(df_bureau['SCORE_02'], errors='coerce')

In [23]:
df_bureau['SCORE_02'].describe()

count    1.289950e+06
mean     6.275538e+02
std      9.607562e+01
min      1.000000e+00
25%      5.570000e+02
50%      6.220000e+02
75%      6.960000e+02
max      9.170000e+02
Name: SCORE_02, dtype: float64

Para o processo de ETL:
- Nome: Score02
- Tipo de dado: float

In [15]:
df_bureau

,SAFRA,FLAG_INSTALACAO,FPD,PROD,flag_mig2,SCORE_01,SCORE_02,NUM_CPF
0,202410,1,0,CMV,PRE,562,636,ZZZZZX7XWY8
1,202410,1,1,CMV,PRE,546,518,ZZZZZX88YXY
2,202410,1,0,CMV,PRE,621,750,ZZZZZYT7XYT
3,202410,1,1,CMV,PRE,609,679,ZZZZZNTXY9Z
4,202410,1,0,CMV,PRE,621,722,ZZZZZ79ZXUX
...,...,...,...,...,...,...,...,...
1290521,202503,1,0,CMV,PRE,604,674,99997YWXNZZ
1290522,202503,1,0,CMV,PRE,688,765,99998TYXZN8
1290523,202503,1,0,CMV,PRE,616,630,9999888YYU9
1290524,202503,1,0,CMV,PRE,627,649,9999889ZN9X


#### 07 - FPD

In [24]:
# Primeiro iremos transformar a coluna FPD para o tipo int
df_bureau['FPD'] = pd.to_numeric(df_bureau['FPD'], errors='coerce')

In [26]:
# Vamos gerar o total de FDP por safra

fpd_safra = df_bureau.groupby(['SAFRA']).agg({'FPD':'sum', 'NUM_CPF':'count'}).rename(columns={'FPD':'TOTAL_FPD', 'NUM_CPF':'TOTAL_CPF'})

# Agora vamos criar uma coluna com a taxa de FPD por safra
fpd_safra['TAXA_FPD'] = fpd_safra['TOTAL_FPD'] / fpd_safra['TOTAL_CPF']

# Exibindo o resultado
fpd_safra

,TOTAL_FPD,TOTAL_CPF,TAXA_FPD
SAFRA,,,
202410,46239,203828,0.226853
202411,56523,227176,0.248807
202412,54536,227985,0.239209
202501,52177,221002,0.236093
202502,45452,203139,0.223748
202503,49269,207396,0.237560


Para o processo de analise:
- Durante todo o periodo analisado, temos uma variacao de nao pagantes em um total de aproximadamente 23% de inadiplencia.

Para o processo de ETL:
- Nome: FDP
- Tipo de dados: Int

#### 08 - NUM_CPF

In [27]:
# Vamos conferir se a coluna NUM_CPF possui CPFs duplicados
df_bureau['NUM_CPF'].duplicated().sum()

18431

Possuimos registros duplicados do numero de CPF, vamos entender como eles se repetem na base

In [28]:
# Conferindo os CPFs duplicados
df_duplicados = df_bureau[df_bureau['NUM_CPF'].duplicated(keep=False)].sort_values(by='NUM_CPF')

df_duplicados

,SAFRA,FLAG_INSTALACAO,FPD,PROD,flag_mig2,SCORE_01,SCORE_02,NUM_CPF
634039,202412,1,0,CMV,PRE,567.0,525.0,77778ZU9Y87
407362,202411,1,1,CMV,PRE,567.0,538.0,77778ZU9Y87
634015,202412,1,0,CMV,PRE,669.0,730.0,777T8X88Z87
856409,202501,1,0,CMV,PRE,669.0,719.0,777T8X88Z87
856380,202501,1,1,CMV,PRE,603.0,705.0,777W777N8WU
...,...,...,...,...,...,...,...,...
431064,202412,1,0,CMV,PRE,645.0,648.0,ZZZZTN7XZ8N
203857,202411,1,0,CMV,PRE,548.0,631.0,ZZZZYNX9XYX
14,202410,1,1,CMV,PRE,548.0,612.0,ZZZZYNX9XYX
879991,202502,1,1,CMV,PRE,2.0,550.0,ZZZZZZZZ787


In [29]:
# Vamos fazer um teste e manter, a partir da base completa, apenas o ultimo registro de cada CPF, para isso iremos ordenar a base pela coluna SAFRA e depois remover os duplicados mantendo o ultimo registro
df_bureau_unique = df_bureau.sort_values(by='SAFRA').drop_duplicates(subset='NUM_CPF', keep='last')

In [36]:
# Total de CPFs duplicados
df_bureau_unique['NUM_CPF'].duplicated().sum()

0

In [37]:
# Total de CPFs únicos na base após remoção de duplicados
df_bureau_unique['NUM_CPF'].count()

1272095

In [39]:
# Total de CPFs únicos na base original
df_bureau['NUM_CPF'].count()

1290526

In [40]:
# Diferenca de CPFs entre a base original e a base sem duplicados
df_bureau['NUM_CPF'].count() - df_bureau_unique['NUM_CPF'].count()

18431

In [41]:
# Procentagem de dados removidos
(df_bureau['NUM_CPF'].count() - df_bureau_unique['NUM_CPF'].count()) / df_bureau['NUM_CPF'].count() * 100

1.4281773478411128